# Famous Landmark Detection using Deep Learning

**Internship Project**

This project builds an image classification model that recognizes famous landmarks/monuments
(e.g., Eiffel Tower, Taj Mahal, Statue of Liberty, Colosseum, etc.) from photographs, using
**Transfer Learning** with a pretrained CNN (MobileNetV2).

### Project Pipeline
1. Setup & Imports
2. Dataset Acquisition
3. Data Exploration & Visualization
4. Data Preprocessing & Augmentation
5. Model Building (Transfer Learning)
6. Model Training
7. Training Results Visualization
8. Model Evaluation (Confusion Matrix, Classification Report)
9. Save the Trained Model
10. Inference on New / Custom Images
11. Conclusion & Future Work

---


## 1. Setup & Imports

Install/import all the libraries needed for this project. If a library is missing, uncomment the
`pip install` line in the first cell.

In [ ]:
# Uncomment if running in a fresh environment (e.g. Google Colab)
# !pip install tensorflow matplotlib seaborn scikit-learn pillow kagglehub -q

import os
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path

import tensorflow as tf
from tensorflow.keras import layers, models
from tensorflow.keras.applications import MobileNetV2
from tensorflow.keras.applications.mobilenet_v2 import preprocess_input
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from sklearn.metrics import classification_report, confusion_matrix

print("TensorFlow version:", tf.__version__)
print("GPU available:", tf.config.list_physical_devices('GPU'))

# Reproducibility
SEED = 42
np.random.seed(SEED)
tf.random.set_seed(SEED)


## 2. Dataset Acquisition

This notebook expects the dataset to be arranged in the standard **Keras `image_dataset_from_directory`**
folder structure, i.e. one sub-folder per landmark class:

```
data/
├── train/
│   ├── eiffel_tower/
│   │   ├── img001.jpg
│   │   └── ...
│   ├── taj_mahal/
│   ├── statue_of_liberty/
│   ├── colosseum/
│   └── great_wall_of_china/
└── test/
    ├── eiffel_tower/
    ├── taj_mahal/
    ├── statue_of_liberty/
    ├── colosseum/
    └── great_wall_of_china/
```

**Where to get landmark image data (pick one):**
- Kaggle: *"Landmark Classification"*, *"Famous Landmarks around the World"*, or the
  *Google Landmarks Dataset v2* (a smaller curated subset is recommended for a student project).
- Build your own mini-dataset: 100–300 images per landmark class scraped/collected manually
  (make sure you have the right to use them).

Download the data, unzip it, and update `DATA_DIR` below to point at the folder containing
`train/` and `test/`. If you only have one folder of images (no train/test split), use the
`validation_split` option shown in the code — no need to make two folders.


In [ ]:
# ---- CONFIG: update this path to your dataset location ----
DATA_DIR = Path("data")          # expects DATA_DIR/train and DATA_DIR/test (optional)
IMG_SIZE = (224, 224)
BATCH_SIZE = 32

# Quick sanity check
if DATA_DIR.exists():
    classes = sorted([d.name for d in (DATA_DIR / "train").iterdir() if d.is_dir()]) \
        if (DATA_DIR / "train").exists() else sorted([d.name for d in DATA_DIR.iterdir() if d.is_dir()])
    print("Found classes:", classes)
else:
    print(f"'{DATA_DIR}' not found yet. Download your dataset and place it here before continuing.")


## 3. Loading the Data

We use `image_dataset_from_directory` which automatically infers class labels from the folder
names and creates a `tf.data.Dataset` (fast, memory-efficient, and works with any dataset size).

If you only have a single folder of images (no separate train/test), set `SINGLE_FOLDER = True`
and this will automatically create an 80/20 train/validation split.

In [ ]:
SINGLE_FOLDER = False   # set True if you don't have separate train/ and test/ folders

if SINGLE_FOLDER:
    train_ds = tf.keras.utils.image_dataset_from_directory(
        DATA_DIR, validation_split=0.2, subset="training", seed=SEED,
        image_size=IMG_SIZE, batch_size=BATCH_SIZE)
    val_ds = tf.keras.utils.image_dataset_from_directory(
        DATA_DIR, validation_split=0.2, subset="validation", seed=SEED,
        image_size=IMG_SIZE, batch_size=BATCH_SIZE)
else:
    train_ds = tf.keras.utils.image_dataset_from_directory(
        DATA_DIR / "train", seed=SEED, image_size=IMG_SIZE, batch_size=BATCH_SIZE)
    val_ds = tf.keras.utils.image_dataset_from_directory(
        DATA_DIR / "test", seed=SEED, image_size=IMG_SIZE, batch_size=BATCH_SIZE)

class_names = train_ds.class_names
num_classes = len(class_names)
print("Classes:", class_names)
print("Number of classes:", num_classes)


## 4. Data Exploration & Visualization

Always look at your data first! Let's plot a sample grid of images with their labels, and check
class balance.

In [ ]:
plt.figure(figsize=(10, 10))
for images, labels in train_ds.take(1):
    for i in range(min(9, len(images))):
        ax = plt.subplot(3, 3, i + 1)
        plt.imshow(images[i].numpy().astype("uint8"))
        plt.title(class_names[labels[i]])
        plt.axis("off")
plt.suptitle("Sample Training Images")
plt.tight_layout()
plt.show()


In [ ]:
# Class distribution (train set)
counts = {name: 0 for name in class_names}
for _, labels in train_ds.unbatch():
    counts[class_names[int(labels)]] += 1

plt.figure(figsize=(8, 4))
sns.barplot(x=list(counts.keys()), y=list(counts.values()))
plt.xticks(rotation=45, ha="right")
plt.ylabel("Number of images")
plt.title("Class Distribution (Training Set)")
plt.tight_layout()
plt.show()


## 5. Data Preprocessing & Augmentation

- **Caching & prefetching** for fast training pipelines.
- **Data augmentation** (random flip, rotation, zoom) to reduce overfitting, since landmark
  datasets are often small.
- **Preprocessing** to match what MobileNetV2 expects (scales pixels to [-1, 1]).

In [ ]:
AUTOTUNE = tf.data.AUTOTUNE

data_augmentation = tf.keras.Sequential([
    layers.RandomFlip("horizontal"),
    layers.RandomRotation(0.1),
    layers.RandomZoom(0.1),
    layers.RandomContrast(0.1),
], name="data_augmentation")

def prepare(ds, training=False):
    if training:
        ds = ds.map(lambda x, y: (data_augmentation(x, training=True), y), num_parallel_calls=AUTOTUNE)
    ds = ds.map(lambda x, y: (preprocess_input(x), y), num_parallel_calls=AUTOTUNE)
    return ds.cache().prefetch(buffer_size=AUTOTUNE)

train_ds_prepped = prepare(train_ds, training=True)
val_ds_prepped = prepare(val_ds, training=False)


## 6. Model Building — Transfer Learning with MobileNetV2

We use **MobileNetV2** pretrained on ImageNet as a frozen feature extractor, and add a small
classification head on top. This works very well even with a few hundred images per class,
which is typical for a landmark-recognition intern project.

In [ ]:
base_model = MobileNetV2(input_shape=IMG_SIZE + (3,), include_top=False, weights="imagenet")
base_model.trainable = False   # freeze the pretrained backbone

inputs = tf.keras.Input(shape=IMG_SIZE + (3,))
x = base_model(inputs, training=False)
x = layers.GlobalAveragePooling2D()(x)
x = layers.Dropout(0.3)(x)
x = layers.Dense(128, activation="relu")(x)
x = layers.Dropout(0.2)(x)
outputs = layers.Dense(num_classes, activation="softmax")(x)

model = models.Model(inputs, outputs, name="landmark_classifier")

model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=1e-3),
    loss="sparse_categorical_crossentropy",
    metrics=["accuracy"]
)

model.summary()


## 7. Model Training

We train in **two phases**:
1. Train only the new classification head (backbone frozen) — fast and stable.
2. **Fine-tune**: unfreeze the top layers of MobileNetV2 with a low learning rate to squeeze out
   extra accuracy.

`EarlyStopping` and `ReduceLROnPlateau` help avoid overfitting and speed up convergence.

In [ ]:
callbacks = [
    tf.keras.callbacks.EarlyStopping(monitor="val_loss", patience=5, restore_best_weights=True),
    tf.keras.callbacks.ReduceLROnPlateau(monitor="val_loss", factor=0.5, patience=2),
]

EPOCHS_HEAD = 10

history = model.fit(
    train_ds_prepped,
    validation_data=val_ds_prepped,
    epochs=EPOCHS_HEAD,
    callbacks=callbacks
)


In [ ]:
# ---- Phase 2: Fine-tuning ----
base_model.trainable = True

# Freeze all layers except the last ~30 (fine-tune only the top of the backbone)
fine_tune_at = len(base_model.layers) - 30
for layer in base_model.layers[:fine_tune_at]:
    layer.trainable = False

model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=1e-5),  # much lower LR for fine-tuning
    loss="sparse_categorical_crossentropy",
    metrics=["accuracy"]
)

EPOCHS_FINE_TUNE = 10

history_fine = model.fit(
    train_ds_prepped,
    validation_data=val_ds_prepped,
    epochs=EPOCHS_FINE_TUNE,
    callbacks=callbacks
)


## 8. Training Results Visualization

Plot accuracy and loss curves across both training phases to check for overfitting/underfitting.

In [ ]:
def combine_history(h1, h2, key):
    return h1.history[key] + h2.history[key]

acc = combine_history(history, history_fine, "accuracy")
val_acc = combine_history(history, history_fine, "val_accuracy")
loss = combine_history(history, history_fine, "loss")
val_loss = combine_history(history, history_fine, "val_loss")

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].plot(acc, label="Train Accuracy")
axes[0].plot(val_acc, label="Val Accuracy")
axes[0].axvline(len(history.history["accuracy"]), color="gray", linestyle="--", label="Fine-tuning starts")
axes[0].set_title("Accuracy over Epochs")
axes[0].set_xlabel("Epoch")
axes[0].legend()

axes[1].plot(loss, label="Train Loss")
axes[1].plot(val_loss, label="Val Loss")
axes[1].axvline(len(history.history["loss"]), color="gray", linestyle="--", label="Fine-tuning starts")
axes[1].set_title("Loss over Epochs")
axes[1].set_xlabel("Epoch")
axes[1].legend()

plt.tight_layout()
plt.show()


## 9. Model Evaluation

Evaluate on the validation/test set with a **confusion matrix** and **classification report**
(precision, recall, F1-score per landmark class).

In [ ]:
y_true = []
y_pred = []

for images, labels in val_ds_prepped:
    preds = model.predict(images, verbose=0)
    y_pred.extend(np.argmax(preds, axis=1))
    y_true.extend(labels.numpy())

print(classification_report(y_true, y_pred, target_names=class_names))

cm = confusion_matrix(y_true, y_pred)
plt.figure(figsize=(8, 6))
sns.heatmap(cm, annot=True, fmt="d", cmap="Blues", xticklabels=class_names, yticklabels=class_names)
plt.xlabel("Predicted")
plt.ylabel("Actual")
plt.title("Confusion Matrix")
plt.tight_layout()
plt.show()


## 10. Save the Trained Model

Save the model so it can be reloaded later (e.g. for a web app or API) without retraining.

In [ ]:
model.save("landmark_classifier.keras")
print("Model saved as landmark_classifier.keras")


## 11. Inference on a New / Custom Image

Use the trained model to predict the landmark in any new photo. Update `IMAGE_PATH` to point
to your own test image.

In [ ]:
def predict_landmark(image_path, model, class_names, img_size=IMG_SIZE, top_k=3):
    img = tf.keras.utils.load_img(image_path, target_size=img_size)
    img_array = tf.keras.utils.img_to_array(img)
    img_array = preprocess_input(img_array)
    img_array = np.expand_dims(img_array, axis=0)

    preds = model.predict(img_array, verbose=0)[0]
    top_indices = preds.argsort()[-top_k:][::-1]

    plt.imshow(img)
    plt.axis("off")
    title = "\n".join([f"{class_names[i]}: {preds[i]*100:.1f}%" for i in top_indices])
    plt.title(title)
    plt.show()

    return [(class_names[i], float(preds[i])) for i in top_indices]

# Example usage — replace with the path to an actual image on your machine
IMAGE_PATH = "sample_test_image.jpg"
if os.path.exists(IMAGE_PATH):
    results = predict_landmark(IMAGE_PATH, model, class_names)
    print(results)
else:
    print(f"Set IMAGE_PATH to a real image file to test inference (currently '{IMAGE_PATH}' not found).")


## 12. Conclusion & Future Work

**Summary:** This notebook implements an end-to-end famous-landmark recognition pipeline using
transfer learning on MobileNetV2 — from data loading and augmentation, through training and
fine-tuning, to evaluation and inference on new images.

**Possible extensions for a stronger submission:**
- Try other backbones (EfficientNetB0, ResNet50) and compare accuracy.
- Add **Grad-CAM** visualizations to show *which part of the image* the model focused on.
- Deploy the model as a simple web app (Streamlit/Flask) where a user uploads a photo and gets
  the predicted landmark.
- Expand the dataset with more landmark classes and images per class for better generalization.
- Add a script to automatically scrape/download landmark images (with correct usage rights) to
  grow the dataset.

---
*Prepared as an internship project — Landmark Detection using Deep Learning.*
